In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from difflib import SequenceMatcher
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "ebm_nlp_2_00").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/ebm_nlp_2_00 from the current notebook directory")

DATA_FILE = PROJECT_ROOT / "ebm_nlp_2_00" / "processed" / "ebm_abstracts_full.npz"
OUTPUT_DIR = PROJECT_ROOT / "task2-llm"
OUTPUT_DIR.mkdir(exist_ok=True)
PREDICTION_COLUMNS = ["method", "doc_id", "text", "Pop_gold", "Pop_pred", "Int_gold", "Int_pred", "Out_gold", "Out_pred"]

class PIOExtraction(BaseModel):
    population: Optional[str] = Field(description="The patients or problem. Return null if none.")
    intervention: Optional[str] = Field(description="The main treatment. Return null if none.")
    outcome: Optional[str] = Field(description="The primary results. Return null if none.")

llm = OllamaLLM(model="llama3.1", format="json", temperature=0)
parser = JsonOutputParser(pydantic_object=PIOExtraction)

def extract_text_from_mask(text, mask):
    words = text.split()
    spans = []
    current = []
    for word, label in zip(words, mask):
        if label == 1:
            current.append(word)
        elif current:
            spans.append(" ".join(current))
            current = []
    if current:
        spans.append(" ".join(current))
    return " ; ".join(spans) if spans else "null"

def normalize_prediction(value):
    if value is None:
        return "null"
    if isinstance(value, list):
        items = [normalize_prediction(v) for v in value]
        items = [v for v in items if v.lower() != "null"]
        return " ; ".join(items) if items else "null"
    if isinstance(value, dict):
        return normalize_prediction(list(value.values()))
    text = str(value).strip()
    if not text or text.lower() in {"null", "none", "nan", "[]"}:
        return "null"
    return text

def evaluate_extraction(original_text, extracted_text, gt_mask, threshold=0.6):
    if not extracted_text or str(extracted_text).lower() in ('null', 'none'):
        return "None", None 
        
    items = [item.strip() for item in str(extracted_text).split(';')]
    
    orig_words = original_text.split()
    item_scores = []
    
    for item in items:
        if not item: continue
        
        ext_words = item.split()
        window = len(ext_words)
        best_ratio, best_start = 0, 0
        
        for i in range(len(orig_words) - window + 1):
            window_text = " ".join(orig_words[i:i+window])
            ratio = SequenceMatcher(None, item.lower(), window_text.lower()).ratio()
            if ratio > best_ratio:
                best_ratio, best_start = ratio, i

        if best_ratio < threshold:
            item_scores.append(0.0) # (Hallucination)
        else:
            mask_slice = gt_mask[best_start : best_start + window]
            if any(label == 1 for label in mask_slice):
                item_scores.append(1.0) # (Hit)
            else:
                item_scores.append(0.0) # (Miss)

    if not item_scores:
        return "None", None

    avg_precision = sum(item_scores) / len(item_scores)
    
    if avg_precision == 1.0:
        return "Hit (Relaxed)", 1.0
    elif avg_precision == 0.0:
        return "Hallucinated", 0.0
    else:
        return "Partial Hit", avg_precision

def run_zero_shot_experiment(test_texts, test_masks, test_ids):
    print(f"\n" + "="*60)
    print(f"RUNNING ZERO-SHOT EXPERIMENT")
    print("="*60)

    template = """You are a medical researcher. Extract PIO elements in JSON format.
    CRITICAL RULE: Your extractions MUST be exact, continuous substrings copied directly from the abstract. Do not add or change any words.
    If a field is not explicitly present, return JSON null for that field. Do not guess.
    
    {format_instructions}
    
    --- ACTUAL TASK ---
    Abstract: {abstract}
    Output:"""

    pipeline = PromptTemplate(
        template=template,
        input_variables=["abstract"], 
        partial_variables={
            "format_instructions": parser.get_format_instructions()
        }
    ) | llm | parser

    limit = min(1000, len(test_texts))
    results = []
    
    for i in tqdm(range(limit), desc="Processing Abstracts (Zero-shot)"):
        text = test_texts[i]
        
        try:
            res = pipeline.invoke({
                "abstract": text
            })
            
            row = {
                "method": "llm_zero_shot",
                "doc_id": test_ids[i],
                "text": text,
                "Pop_gold": extract_text_from_mask(text, test_masks["Pop"][i]),
                "Int_gold": extract_text_from_mask(text, test_masks["Int"][i]),
                "Out_gold": extract_text_from_mask(text, test_masks["Out"][i]),
            }
            
            for short_key, full_key in [('Pop', 'population'), ('Int', 'intervention'), ('Out', 'outcome')]:
                extracted = res.get(full_key)
                row[f"{short_key}_pred"] = normalize_prediction(extracted)
                
            results.append(row)
                
        except Exception as e:
            print(f"\nError processing sample {i}: {e}") 

    if results:
        df = pd.DataFrame(results)
        filename = "llm_zero_shot_predictions.csv"
        df = df.reindex(columns=PREDICTION_COLUMNS)
        df.to_csv(OUTPUT_DIR / filename, index=False)

print("Loading dataset...")
data = np.load(DATA_FILE, allow_pickle=True)
    
test_texts = data['test_texts']
test_masks = {'Pop': data['test_p'], 'Int': data['test_i'], 'Out': data['test_o']}
test_ids = list(data['test_ids']) if 'test_ids' in data.files else [f"test_{i}" for i in range(len(test_texts))]

run_zero_shot_experiment(test_texts, test_masks, test_ids)


Loading dataset...

RUNNING ZERO-SHOT EXPERIMENT


Processing Abstracts (Zero-shot): 100%|██████████| 184/184 [06:32<00:00,  2.13s/it]


In [2]:
import pandas as pd

def calculate_metrics(file_paths):
    all_summaries = []
    
    for path in file_paths:
        try:
            df = pd.read_csv(OUTPUT_DIR / path)
            
            summary = {"Experiment": df['method'].iloc[0] if 'method' in df else path}
            
            for pio in ['Pop', 'Int', 'Out']:
                pred = df[f'{pio}_pred'].fillna('null').astype(str).str.lower()
                gold = df[f'{pio}_gold'].fillna('null').astype(str).str.lower()
                summary[f"{pio}_Coverage"] = f"{(pred != 'null').mean()*100:.1f}%"
                summary[f"{pio}_Gold_Coverage"] = f"{(gold != 'null').mean()*100:.1f}%"
                
            all_summaries.append(summary)
        except Exception as e:
            print(f"Error processing {path}: {e}")
            
    return pd.DataFrame(all_summaries)

my_files = [
    "llm_zero_shot_predictions.csv", 
]

report_table = calculate_metrics(my_files)
print(report_table.to_string(index=False))

   Experiment Pop_Coverage Pop_Gold_Coverage Int_Coverage Int_Gold_Coverage Out_Coverage Out_Gold_Coverage
llm_zero_shot        98.9%            100.0%        95.7%             98.9%        96.7%             97.8%
